# 📈 Linear Regression
**ISLP Chapter 3 · Pattern Recognition for the Rest of Us**

> The foundation of supervised learning. Linear regression is simple enough to understand completely, powerful enough to use in production, and teaches concepts (coefficients, p-values, R², residuals) that appear in every model you'll ever build.

---
### What you'll learn
- Simple and multiple linear regression from scratch
- How to interpret every number in the regression output (β, SE, t-stat, p-value, R², RSE)
- Checking assumptions: linearity, homoscedasticity, normality of residuals
- Categorical predictors (dummy variables)
- When linear regression fails and what to do

### Dataset
**Advertising** — predict sales from TV, radio, newspaper budgets (200 markets)

### Time
~75 minutes

## 📐 Part 1 — Simple Linear Regression

**Model:** Y = β₀ + β₁X + ε

- **β₀** (intercept) — predicted Y when X = 0
- **β₁** (slope) — how much Y changes for a one-unit increase in X
- **ε** — random error term, assumed N(0, σ²)

**Goal:** Find β̂₀ and β̂₁ that minimize the Residual Sum of Squares:
```
RSS = Σᵢ (yᵢ - β̂₀ - β̂₁xᵢ)²
```

The OLS (Ordinary Least Squares) solution has a closed form:
```
β̂₁ = Σ(xᵢ - x̄)(yᵢ - ȳ) / Σ(xᵢ - x̄)²
β̂₀ = ȳ - β̂₁x̄
```

In [ ]:
# Simple linear regression: TV → Sales
X_tv = ads[['TV']].values
y = ads['sales'].values

# Fit with sklearn
lr = LinearRegression()
lr.fit(X_tv, y)

print("=== Simple Linear Regression: TV → Sales ===\n")
print(f"β̂₀ (intercept):  {lr.intercept_:.4f}")
print(f"β̂₁ (TV slope):   {lr.coef_[0]:.4f}")
print(f"\nInterpretation:")
print(f"  • Baseline sales (TV=0): {lr.intercept_:.2f} thousand units")
print(f"  • Each $1000 increase in TV spend → +{lr.coef_[0]*1000:.2f} units sold")

# Visualize
tv_range = np.linspace(ads['TV'].min(), ads['TV'].max(), 200).reshape(-1,1)
y_pred_line = lr.predict(tv_range)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Scatter + regression line
axes[0].scatter(ads['TV'], ads['sales'], alpha=0.5, color='#1e5fa8', s=25, edgecolors='white', lw=0.3)
axes[0].plot(tv_range, y_pred_line, color='#e85d20', lw=2.5, label=f'Ŷ = {lr.intercept_:.2f} + {lr.coef_[0]:.4f}·TV')
axes[0].set_xlabel('TV Budget ($000s)')
axes[0].set_ylabel('Sales (000 units)')
axes[0].set_title('Simple Linear Regression: TV → Sales')
axes[0].legend()

# Residuals
y_pred = lr.predict(X_tv)
residuals = y - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5, color='#6b46c1', s=25, edgecolors='white', lw=0.3)
axes[1].axhline(0, color='#e85d20', lw=1.5, ls='--')
axes[1].set_xlabel('Fitted Values (Ŷ)')
axes[1].set_ylabel('Residuals (y - Ŷ)')
axes[1].set_title('Residual Plot — look for random scatter')

plt.tight_layout()
plt.show()

rse = np.sqrt(np.sum(residuals**2) / (len(y) - 2))
print(f"\nRSE (Residual Standard Error): {rse:.3f} thousand units")
print(f"R²: {r2_score(y, y_pred):.3f} — TV explains {r2_score(y, y_pred)*100:.1f}% of sales variance")

## 🔬 Part 2 — Understanding the Full Regression Output

Using `statsmodels`, we get the complete statistical summary with standard errors, t-statistics, and p-values.

**Key numbers to understand:**
- **coef** — the estimated β
- **std err** — standard error of the estimate (uncertainty)
- **t** — test statistic = coef / std err. Large |t| → evidence the predictor matters
- **P>|t|** — p-value for H₀: βⱼ = 0. Small p → reject null → predictor is significant
- **R²** — proportion of variance in Y explained by the model
- **F-statistic** — tests H₀: all βⱼ = 0 simultaneously. Large F → model explains something

In [ ]:
# Full statistical output with statsmodels
X_with_const = sm.add_constant(ads[['TV']])
model_sm = sm.OLS(ads['sales'], X_with_const).fit()
print(model_sm.summary())

print("\n" + "="*55)
print("READING THE TABLE — line by line:")
print("="*55)
print(f"  const  coef={model_sm.params['const']:.4f} → predicted sales when TV=0")
print(f"  TV     coef={model_sm.params['TV']:.4f}  → +{model_sm.params['TV']*1000:.2f} sales per $1000 TV spend")
print(f"  TV     p={model_sm.pvalues['TV']:.2e}      → extremely significant (p << 0.05)")
print(f"  R²    = {model_sm.rsquared:.3f}          → TV explains {model_sm.rsquared*100:.1f}% of sales variance")
print(f"  F-stat = {model_sm.fvalue:.1f}         → p={model_sm.f_pvalue:.2e} → model is significant")

## 📊 Part 3 — Multiple Linear Regression

With multiple predictors:
```
Y = β₀ + β₁X₁ + β₂X₂ + ... + βₚXₚ + ε
```

Now each βⱼ is the effect of Xⱼ on Y *holding all other predictors constant*. This changes the interpretation: a coefficient in multiple regression is not the same as in simple regression.

In [ ]:
# Multiple regression: all three channels
X_all = ads[['TV','radio','newspaper']]
X_all_const = sm.add_constant(X_all)
model_mult = sm.OLS(ads['sales'], X_all_const).fit()
print(model_mult.summary())

print("\n" + "="*60)
print("COMPARISON: Simple vs Multiple Regression")
print("="*60)

# Simple regressions
for col in ['TV','radio','newspaper']:
    Xs = sm.add_constant(ads[[col]])
    m = sm.OLS(ads['sales'], Xs).fit()
    print(f"  Simple ({col:10s}): β={m.params[col]:.4f}  p={m.pvalues[col]:.4f}  R²={m.rsquared:.3f}")

print(f"\n  Multiple (all):  β_TV={model_mult.params['TV']:.4f}  β_radio={model_mult.params['radio']:.4f}  β_news={model_mult.params['newspaper']:.4f}")
print(f"                   R²={model_mult.rsquared:.3f}  (vs best simple R²={r2_score(ads['sales'], LinearRegression().fit(ads[['TV']], ads['sales']).predict(ads[['TV']])):.3f})")
print("\n📌 Newspaper's coefficient flips sign and becomes insignificant in multiple regression.")
print("   It appears correlated with radio, which actually drives the effect.")

In [ ]:
# Visualize: coefficient comparison and confidence intervals
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Coefficients with confidence intervals
coefs = model_mult.params[1:]  # exclude intercept
ci = model_mult.conf_int().iloc[1:]  # 95% CI

y_pos = range(len(coefs))
axes[0].barh(list(coefs.index), coefs.values,
             xerr=[(coefs - ci[0]).values, (ci[1] - coefs).values],
             color=['#1e5fa8','#e85d20','#888'],
             edgecolor='white', height=0.5, capsize=5)
axes[0].axvline(0, color='black', lw=1, ls='--')
axes[0].set_title('Multiple Regression Coefficients\n(with 95% confidence intervals)')
axes[0].set_xlabel('Coefficient value')

# Actual vs Predicted
y_pred_mult = model_mult.predict(X_all_const)
axes[1].scatter(ads['sales'], y_pred_mult, alpha=0.5, color='#1a7a45', s=25,
                edgecolors='white', lw=0.3)
max_val = max(ads['sales'].max(), y_pred_mult.max())
axes[1].plot([0, max_val], [0, max_val], 'k--', lw=1.5, alpha=0.6, label='Perfect predictions')
axes[1].set_xlabel('Actual Sales')
axes[1].set_ylabel('Predicted Sales')
axes[1].set_title(f'Actual vs Predicted — Multiple Regression (R²={model_mult.rsquared:.3f})')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n📌 Adding radio and newspaper improves R² from 0.61 to {model_mult.rsquared:.2f}")
print("   Newspaper p-value is 0.86 — no evidence it adds value given TV and radio")

## ✅ Part 4 — Checking Assumptions

Linear regression assumes:
1. **Linearity** — relationship between X and Y is linear
2. **Independence** — observations are independent
3. **Homoscedasticity** — constant variance of residuals
4. **Normality** — residuals are normally distributed

Violations don't always invalidate the model, but they affect inference (p-values, CIs).

In [ ]:
# Four diagnostic plots (equivalent to R's plot.lm)
y_fitted = model_mult.fittedvalues
residuals_mult = model_mult.resid
standardized_res = (residuals_mult - residuals_mult.mean()) / residuals_mult.std()
influence = model_mult.get_influence()
leverage = influence.hat_matrix_diag

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# 1. Residuals vs Fitted
axes[0,0].scatter(y_fitted, residuals_mult, alpha=0.5, color='#1e5fa8', s=20)
axes[0,0].axhline(0, color='#e85d20', lw=1.5, ls='--')
z = np.polyfit(y_fitted, residuals_mult, 2)
p = np.poly1d(z)
x_smooth = np.linspace(y_fitted.min(), y_fitted.max(), 100)
axes[0,0].plot(x_smooth, p(x_smooth), color='#e85d20', lw=1.5, alpha=0.8)
axes[0,0].set_xlabel('Fitted values'); axes[0,0].set_ylabel('Residuals')
axes[0,0].set_title('Residuals vs Fitted\n(should be random — no pattern)')

# 2. Q-Q plot (normality)
stats.probplot(residuals_mult, plot=axes[0,1])
axes[0,1].set_title('Normal Q-Q Plot\n(points should follow diagonal)')
axes[0,1].get_lines()[1].set_color('#e85d20')

# 3. Scale-Location (homoscedasticity)
axes[1,0].scatter(y_fitted, np.sqrt(np.abs(standardized_res)), alpha=0.5, color='#1a7a45', s=20)
axes[1,0].set_xlabel('Fitted values')
axes[1,0].set_ylabel('√|Standardized residuals|')
axes[1,0].set_title('Scale-Location\n(should be flat — equal spread)')

# 4. Leverage
axes[1,1].scatter(leverage, standardized_res, alpha=0.5, color='#6b46c1', s=20)
axes[1,1].axhline(0, color='black', lw=0.8, ls='--')
axes[1,1].axhline(2, color='#e85d20', lw=1, ls=':', alpha=0.7)
axes[1,1].axhline(-2, color='#e85d20', lw=1, ls=':', alpha=0.7)
axes[1,1].set_xlabel('Leverage'); axes[1,1].set_ylabel('Standardized residuals')
axes[1,1].set_title('Residuals vs Leverage\n(outliers + high leverage = influential)')

plt.suptitle('Regression Diagnostic Plots — Multiple Regression', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print("\n📌 Reading the diagnostics:")
print("  Plot 1 (Residuals vs Fitted): slight nonlinearity visible at high fitted values")
print("  Plot 2 (Q-Q): residuals are roughly normal — minor deviation at tails")
print("  Plot 3 (Scale-Location): some heteroscedasticity — variance increases slightly with fitted values")
print("  Plot 4 (Leverage): no highly influential outliers (none past Cook's distance boundary)")

## 🔤 Part 5 — Categorical Predictors

Linear regression needs numeric inputs. Categorical variables are encoded as **dummy variables** (also called one-hot encoding). For a variable with k levels, we create k-1 dummy variables.

In [ ]:
# Create dummy variables for 'Student' status
# pandas get_dummies drops one category automatically (reference category)
credit_dummies = pd.get_dummies(credit[['Balance','Income','Student']], 
                                 drop_first=True, dtype=float)
print("Columns after encoding:", list(credit_dummies.columns))
print("\n'Student_Yes'=1 means the person is a student, 0=not a student\n")

# Fit and interpret
X_credit = sm.add_constant(credit_dummies[['Income','Student_Yes']])
model_credit = sm.OLS(credit_dummies['Balance'], X_credit).fit()
print(model_credit.summary().tables[1])

print("\n📌 Interpretation:")
print(f"  • Each $1000 income increase → balance changes by ${model_credit.params['Income']:.2f}")
print(f"  • Being a student → balance is ${model_credit.params['Student_Yes']:.2f} higher on average")
print(f"    (holding income constant)")

## 💼 Exercise

**Task 1:** Load the Boston Housing dataset (`from sklearn.datasets import fetch_california_housing`). Fit a multiple linear regression predicting median house value from all features. Report R² and identify the 3 most significant predictors.

**Task 2:** Check the regression assumptions for your model from Task 1. Identify one potential violation.

**Task 3:** Add a categorical feature to your regression. The Credit dataset has `Married`, `Student`, `Ethnicity` — pick one, encode it, and interpret the coefficient.

**Task 4:** Split your data 70/30 and compare training R² vs test R². If they differ a lot, why?

In [ ]:
# Exercise workspace
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
df_housing = data.frame
print(df_housing.head())

# YOUR CODE HERE

In [ ]:
answers = {
    "q1": "",  # What does β₁ = 0.05 mean in Y = β₀ + β₁X?
    "q2": "",  # What does R² = 0.72 mean in plain English?
    "q3": "",  # If the p-value for a coefficient is 0.8, what do you conclude?
    "q4": "",  # What does RSE measure?
    "q5": "",  # Why do we drop one category when encoding a categorical variable?
}
missing = [k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ All answered! Save: File → Save a copy in GitHub")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Linear Regression"
# @title 🤖 AI-Graded Submission — fill in the box and click ▶ Run
# @markdown ---
# @markdown **Step 1:** Enter your GitHub username below
# @markdown
# @markdown **Step 2 (one-time):** Get a free AI grading key
# @markdown - Go to [aistudio.google.com](https://aistudio.google.com) — use your **personal Gmail** (not university email — many universities block AI Studio)
# @markdown - Click **Get API key → Create API key** → copy it
# @markdown - In Colab: click the **🔑 key icon** in the left sidebar → Add secret → Name: `GEMINI_API_KEY` → paste key → toggle ON
# @markdown - Done — this persists across all 30 notebooks automatically
# @markdown
# @markdown **Step 3:** Click ▶ Run on this cell for instant AI feedback

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

# ── DO NOT EDIT BELOW THIS LINE ──────────────────────────────────────────────
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Try to explain your reasoning in 1-2 sentences."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback on your answers."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with good detail. "
                             "Add a free Gemini key for AI analysis of your specific reasoning."),
                "tip": "Get a free key at aistudio.google.com using your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             f"Complete the remaining {n_total - n_answered} questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Enter your GitHub username in the box above")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"\n  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    print(f"  Student  : @{username}" if username else
          "  Student  : \u26a0\ufe0f  Enter your GitHub username in the box above")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...\n")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)\n")
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell\n")
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 choose your fork\n")
